In [5]:
# %%
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import re
import random

# Ensure project root is on PYTHONPATH
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# %%
# 8.1) Imports
from src.io.load_biosecurid import load_local
from src.nn_utils.data import resample_sequence
from keras.models import load_model, Model as KModel
from src.keras_layers.diff_dtw import DiffDTW
from src.evaluation.evaluation import dtw_distance
from typing import cast

# %%
# 8.2) Paths and constants
PROC_ROOT = project_root / "data" / "processed"
MODEL_DIR = project_root / "models"
SEQ_LEN    = 100   # must match training
N_FEATURES = 9

# Helper: extract user ID and signature index from filename
sig_re = re.compile(r"u(\d{4})s\d{4}_sg(\d{4})")
def extract_ids(fp: Path):
    m = sig_re.search(fp.name)
    if not m:
        return None, None
    return m.group(1), int(m.group(2))

# %%
# 8.3) Load the trained Siamese-DTW model
model_path = MODEL_DIR / "siamese_final.keras"
assert model_path.exists(), f"Model file not found: {model_path}"
raw_model = load_model(
    str(model_path),
    custom_objects={"DiffDTW": DiffDTW}  # type: ignore
)
siamese: KModel = cast(KModel, raw_model)
print("Loaded Siamese-DTW model:")
siamese.summary()

# %%
# %%
# 8.4) Gather signature files and build test scenarios
all_files = list(PROC_ROOT.rglob("*/LocalFunctions/*.mat"))
# groups maps user ID → list of (Path, signature_index)
groups: dict[str, list[tuple[Path, int]]] = {}
for f in all_files:
    uid, sg = extract_ids(f)
    if uid and sg is not None:
        groups.setdefault(uid, []).append((f, sg))

# Define which signature indices are considered genuine vs skilled forgery
GENUINE_SG = {1, 2, 6, 7}
SKILLED_SG = {3, 4, 5}

# Build test_scenarios: list of (user, case, reference_path, target_path)
test_scenarios: list[tuple[str, str, Path, Path]] = []
for uid, entries in groups.items():
    # entries: list of (Path, sg_index)
    # select a reference genuine signature
    genuine_entries = [entry for entry in entries if entry[1] in GENUINE_SG]
    if not genuine_entries:
        continue
    ref_path, _ = genuine_entries[0]
    # genuine targets: other genuine signatures for this user
    for path, sg in entries:
        if path != ref_path and sg in GENUINE_SG:
            test_scenarios.append((uid, "genuine", ref_path, path))
    # skilled forgery targets for this user
    for path, sg in entries:
        if sg in SKILLED_SG:
            test_scenarios.append((uid, "skilled", ref_path, path))
    # random forgery: pick one signature from a different user
    other_entries = [(p, s) for ouid, entr in groups.items() if ouid != uid for (p, s) in entr]
    if other_entries:
        rand_path, _ = random.choice(other_entries)
        test_scenarios.append((uid, "random", ref_path, rand_path))

# %%
# 8.5) Run tests and collect results
# (a) Pre-load & resample each file just once
unique_fps = {fp for _,_,fp1,fp2 in test_scenarios for fp in (fp1, fp2)}
seq_cache = {
    fp: resample_sequence(load_local(fp), SEQ_LEN)
    for fp in unique_fps
}

# (b) Build NumPy batches of shape (M, 100, 9)
M = len(test_scenarios)
refs = np.stack([seq_cache[ref] for _,_,ref,_ in test_scenarios])
tgts = np.stack([seq_cache[tgt] for _,_,_,tgt in test_scenarios])

# (c) Batch predict with Siamese-DTW
#    siamese is your loaded KModel from cell 8.3
probs = siamese.predict([refs, tgts], batch_size=32)[:, 1]

# (d) Compute classic DTW distances in a vectorized loop
dists = np.array([dtw_distance(refs[i], tgts[i]) for i in range(M)])

# (e) Assemble your results DataFrame
df_results = pd.DataFrame({
    "user":       [uid   for uid,_,_,_ in test_scenarios],
    "case":       [case  for _,case,_,_ in test_scenarios],
    "ref":        [ref.name for _,_,ref,_ in test_scenarios],
    "target":     [tgt.name for _,_,_,tgt in test_scenarios],
    "siam_score": np.round(probs, 4),
    "dtw_dist":   np.round(dists, 2),
    "true_label": [1 if case=="genuine" else 0 for _,case,_,_ in test_scenarios]
})

# (f) Display
from IPython.display import display
display(df_results)


Loaded Siamese-DTW model:


Model: "SiameseDTW"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inputA (InputLayer) │ (None, 100, 9)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inputB (InputLayer) │ (None, 100, 9)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ td_dense_0          │ (None, 100, 7)    │         70 │ inputA[0][0],     │
│ (TimeDistributed)   │                   │            │ inputB[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ td_dense_1          │ (None, 100, 5)    │         40 │ td_dense_0[0][0], │
│ (TimeDistributed)   │                   │            │ td_dense_0[1][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ diffdtw (DiffDTW)   │ (None, 1)         │          0 │ td_dense_1[0][0], │
│                     │                   │            │ td_dense_1[1][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ post_dtw_dense_0    │ (None, 16)        │         32 │ diffdtw[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ post_dtw_dense_1    │ (None, 8)         │        136 │ post_dtw_dense_0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 2)         │         18 │ post_dtw_dense_1… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 890 (3.48 KB)

 Trainable params: 296 (1.16 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 594 (2.32 KB)

350/350 ━━━━━━━━━━━━━━━━━━━━ 250s 709ms/step


,user,case,ref,target,siam_score,dtw_dist,true_label
0,1001,genuine,u1001s0001_sg0001.mat,u1001s0001_sg0002.mat,0.9938,125.47,1
1,1001,genuine,u1001s0001_sg0001.mat,u1001s0001_sg0006.mat,0.9945,137.63,1
2,1001,genuine,u1001s0001_sg0001.mat,u1001s0001_sg0007.mat,0.9955,134.89,1
3,1001,genuine,u1001s0001_sg0001.mat,u1001s0002_sg0001.mat,0.9936,138.04,1
4,1001,genuine,u1001s0001_sg0001.mat,u1001s0002_sg0002.mat,0.9942,155.58,1
...,...,...,...,...,...,...,...
11195,1400,skilled,u1400s0001_sg0001.mat,u1400s0003_sg0005.mat,0.0293,305.59,0
11196,1400,skilled,u1400s0001_sg0001.mat,u1400s0004_sg0003.mat,0.0148,322.64,0
11197,1400,skilled,u1400s0001_sg0001.mat,u1400s0004_sg0004.mat,0.0348,313.91,0
11198,1400,skilled,u1400s0001_sg0001.mat,u1400s0004_sg0005.mat,0.0280,294.93,0
